## 🧬 NGS Mini‑Pipeline Training

## 🔍 Looking for variants

We have mapped our reads to the reference genome and produced a sorted, indexed BAM file.

The next question is:

👉 *Are there any positions in the genome where our sample differs from the reference?*

This process is called **variant calling**.

---

### 🧠 What is a variant?

A **variant** is a position in the genome where the DNA sequence of a sample differs from the reference genome.

The most common type is a **SNV (Single Nucleotide Variant)** — a single base change:

```
Reference:  ...ACGTACGT...
Sample:     ...ACGTATGT...  ← T instead of A at this position
```

Other types include:
- **Insertions** — extra bases present in the sample
- **Deletions** — bases missing from the sample
- **MNVs** — multiple consecutive base changes

---

### 🧪 Our training dataset

Our synthetic FASTQ files were generated from a **deliberately mutated reference**.

Before generating the reads, a single SNV was introduced at **position 201** of our mini reference (`chr1:100000-101000`). This simulates a patient sample carrying a known variant.

When we map those reads back to the **original reference** and run the variant caller, it should detect exactly that position.

This gives us a controlled experiment: we know the answer before we run the analysis.

---

### ⚙️ Tools used in this notebook

| Tool | Purpose |
|---|---|
| **bcftools mpileup** | Summarises the bases seen at each position across all reads |
| **bcftools call** | Identifies positions that differ significantly from the reference |

---

## 🔬 How does variant calling work?

The process has two main steps:

**Step 1 — Pileup**
At every position in the reference, `bcftools mpileup` looks at all reads covering that position and records which base each read has.

```
Reference position 201:  A
Read 1:                  T
Read 2:                  T
Read 3:                  T
Read 4:                  T
                         ↑ all reads show T → likely a real variant
```

**Step 2 — Calling**
`bcftools call` applies a statistical model to the pileup data and decides whether the observed difference is a true variant or sequencing noise.

The output is a **VCF file** (Variant Call Format) — we will explore that in the next notebook.

---

## ▶️ Running the variant caller

### Step 1 — Generate the pileup with `bcftools mpileup`

| Flag | Meaning |
|---|---|
| `-f` | Reference FASTA file |
| `-o` | Output file |

The output is a BCF file (binary version of VCF) containing the raw pileup data at every covered position.

In [ ]:
!bcftools mpileup -f ../ref_data/mini_reference.fa ../fastq_files/aligned_sorted.bam -o ../fastq_files/pileup.bcf

### Step 2 — Call variants with `bcftools call`

| Flag | Meaning |
|---|---|
| `-m` | Use the multiallelic calling model (recommended for most use cases) |
| `-v` | Output only variant sites (skip positions identical to the reference) |
| `-o` | Output VCF file |

In [ ]:
!bcftools call -mv ../fastq_files/pileup.bcf -o ../fastq_files/variants.vcf

### Step 3 — View the results

We use `grep -v "^##"` to skip the long header lines and show only the column header and variant entries.

In [ ]:
!grep -v "^##" ../fastq_files/variants.vcf

---

## 🧠 Interpreting the output

You should see one or more variant lines. Each line represents a position where the sample differs from the reference.

| Column | Name | Meaning |
|---|---|---|
| 1 | CHROM | Contig/chromosome name |
| 2 | POS | Position in the reference (1-based) |
| 3 | ID | Variant identifier (`.` = none assigned) |
| 4 | REF | Reference base at this position |
| 5 | ALT | Alternative base observed in the sample |
| 6 | QUAL | Quality score of the variant call |
| 7 | FILTER | Filter status (`PASS` = passed all filters) |
| 8 | INFO | Additional statistics |
| 9 | FORMAT | Format of the sample column |
| 10 | SAMPLE | Genotype and per-sample statistics |

> 👉 Is the variant at position **201**? That is where we introduced the SNV in `create_data.py`.
> 👉 What is the REF base and what is the ALT base?
> 👉 What does the QUAL score tell you about confidence in this call?

In the next notebook we will explore the VCF format in much more detail.

---

## 🏥 Variant calling in a clinical pipeline

The command we used is a minimal example. In a real clinical pipeline, variant calling is more complex:

- Multiple variant callers are often used and their results compared
- Variants are filtered by quality, depth, allele frequency, and strand bias
- Hard filters or VQSR (Variant Quality Score Recalibration) are applied
- Results are annotated with databases (gnomAD, ClinVar, OMIM) before clinical interpretation

The VCF format is the standard output regardless of which caller is used — which is why understanding it is essential.

## About the Author

<p align="center">
  <img src="https://raw.githubusercontent.com/Manuel-DominguezCBG/ngs-teaching-binder/main/assets/Manolo.jpg" width="120" style="border-radius: 6px;">
</p>

**Manuel — Bioinformatician**

Manuel works as a bioinformatician in clinical genomics. He created this resource to support trainees taking their first steps in NGS bioinformatics, with the aim of making the learning process a little less daunting.

Feedback, adaptations, and contributions to this teaching resource are welcome.

**LinkedIn:** [Manuel J. Domínguez](https://www.linkedin.com/in/manuel-j-dom%C3%ADnguez-aa97981b2/)  
**GitHub Repository:** [ngs‑teaching‑binder](https://github.com/Manuel-DominguezCBG/ngs-teaching-binder)

**Huge thanks to the Bioinformatics team and all my WGLS colleagues for helping me make these learning resources better.**